# Real-World Validation

Runs all four approaches on two real-world datasets:
- **Flight Passengers** (1D time series, 4-qubit parallel circuit)
- **California Housing** (8D regression, 8-qubit serial circuit with hardware-efficient ansatz)

Produces `fig:r2s_realworld` and the unified real-world results table.

In [1]:
# ── Smoke test flag ────────────────────────────────────────────
SMOKE_TEST = True

from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import optax
import pennylane as qml
import pandas as pd
import pickle
import sys
import ssl
import urllib.request
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import fetch_california_housing
import seaborn as sns

ssl._create_default_https_context = ssl._create_unverified_context

sys.path.append("..")
from paper_style import apply_paper_style, BLUE, RED, GREEN, ORANGE, PURPLE, APPROACH_COLORS
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

jax.config.update("jax_enable_x64", True)
apply_paper_style()
SEED = 42
np.random.seed(SEED)

## Common hyperparameters

In [2]:
NUM_RUNS       = 2 if SMOKE_TEST else 10
MAX_STEPS      = 50  if SMOKE_TEST else 5000
LR             = 0.001
BATCH_SIZE     = 40
TRAINABLE_BLOCKS = 1
NUM_SU_GATES   = 1

OUT_DIR = Path("results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def weights_ones(n_serial, t_blocks, n_su, n_params):
    return np.ones((n_serial, t_blocks, n_su, n_params))

def square_loss(targets, predictions):
    return 0.5 * sum((t-p)**2 for t,p in zip(targets, predictions)) / len(targets)

print(f"SMOKE_TEST={SMOKE_TEST}  runs={NUM_RUNS}  steps={MAX_STEPS}")

SMOKE_TEST=True  runs=2  steps=50


## Part 1: Flight Passengers
4-qubit parallel circuit, 1 feature map per qubit, SU(16) ansatz (255 params per block)

In [3]:
# ── Data preparation ───────────────────────────────────────────
flights   = sns.load_dataset("flights")
y_raw     = np.array(flights["passengers"], dtype=float)
x_raw     = np.arange(len(y_raw), dtype=float)

scaler_x  = MinMaxScaler(feature_range=(-np.pi, np.pi))
scaler_y  = MinMaxScaler(feature_range=(-1, 1))
x_scaled  = scaler_x.fit_transform(x_raw.reshape(-1, 1)).ravel()
y_scaled  = scaler_y.fit_transform(y_raw.reshape(-1, 1)).ravel()

# Sequential 80/20 split (no shuffling — respect temporal ordering)
split     = int(len(x_scaled) * 0.8)
x_tr, x_te = jnp.array(x_scaled[:split]), jnp.array(x_scaled[split:])
y_tr, y_te = jnp.array(y_scaled[:split]), jnp.array(y_scaled[split:])

print(f"Flight Passengers: {len(x_tr)} train, {len(x_te)} test samples")

Flight Passengers: 115 train, 29 test samples


In [4]:
# ── Circuit ────────────────────────────────────────────────────
FP_WIRES          = 4
FP_SERIAL_LAYERS  = 2
FP_ROT_PARAMS     = 255  # SU(16)

def S_1d(alpha, x, num_wires, enc_layer):
    for w in range(num_wires):
        qml.RX(alpha[w] * x, wires=w)

def W_SU_n(theta, t_blocks, wires_list):
    for i in range(t_blocks):
        qml.SpecialUnitary(theta[i][0], wires=wires_list)

dev_fp = qml.device("default.qubit", wires=FP_WIRES)

@qml.qnode(dev_fp, interface="jax")
def circuit_fp(alpha, weights_SU, x=None):
    W_SU_n(weights_SU[0], TRAINABLE_BLOCKS, list(range(FP_WIRES)))
    for i in range(FP_SERIAL_LAYERS - 1):
        S_1d(alpha[i], x, FP_WIRES, i)
        W_SU_n(weights_SU[i+1], TRAINABLE_BLOCKS, list(range(FP_WIRES)))
    return qml.expval(qml.PauliZ(wires=FP_WIRES-1))

circuit_fp_jit = jax.jit(circuit_fp)

# Approach configurations for Flight Passengers
FP_APPROACHES = [
    ("Fixed Unary",      [[1.0, 1.0, 1.0, 1.0]],   True),
    ("Trainable Unary",  [[1.01, 1.02, 1.03, 1.04]], False),
    ("Fixed Ternary",    [[1.0, 3.0, 9.0, 27.0]],    True),
    ("Trainable Ternary",[[1.0, 3.0, 9.0, 27.0]],    False),
]

In [5]:
fp_results = []

for label, alpha_init, fixed in FP_APPROACHES:
    print(f"\n── Flight Passengers  {label} ──")
    for run in range(NUM_RUNS):
        alpha   = jnp.array(alpha_init)
        weights = jnp.array(weights_ones(
            FP_SERIAL_LAYERS, TRAINABLE_BLOCKS, NUM_SU_GATES, FP_ROT_PARAMS
        ))

        if fixed:
            params = {"weights_SU": weights}
            def cost_fn(p, x, y):
                return square_loss(y, [circuit_fp_jit(alpha, p["weights_SU"], x_)
                                       for x_ in x]).squeeze()
        else:
            params = {"weights_SU": weights, "alpha": alpha}
            def cost_fn(p, x, y):
                return square_loss(y, [circuit_fp_jit(p["alpha"], p["weights_SU"], x_)
                                       for x_ in x]).squeeze()

        opt       = optax.adam(LR)
        opt_state = opt.init(params)

        @jax.jit
        def update(p, s, x, y):
            loss, g = jax.value_and_grad(cost_fn)(p, x, y)
            u, s2   = opt.update(g, s, p)
            return optax.apply_updates(p, u), s2, loss

        for step in range(MAX_STEPS):
            idx = np.random.randint(0, len(x_tr), BATCH_SIZE)
            params, opt_state, c = update(
                params, opt_state, x_tr[idx], y_tr[idx]
            )

        fa = params.get("alpha", alpha)
        preds = [float(circuit_fp_jit(fa, params["weights_SU"], x_)) for x_ in x_te]
        r2    = r2_score(y_te, preds)
        print(f"  run={run+1}  R²={r2:.4f}")
        fp_results.append({"Dataset": "FlightPassengers", "Approach": label,
                            "Run": run, "R2_Score": r2})

fp_df = pd.DataFrame(fp_results)
fp_df.to_csv(OUT_DIR / "flight_passengers.csv", index=False)


── Flight Passengers  Fixed Unary ──
  run=1  R²=-8.0633
  run=2  R²=-8.5555

── Flight Passengers  Trainable Unary ──
  run=1  R²=-8.4677
  run=2  R²=-9.3958

── Flight Passengers  Fixed Ternary ──
  run=1  R²=-2.4012
  run=2  R²=-2.5369

── Flight Passengers  Trainable Ternary ──
  run=1  R²=-2.0929
  run=2  R²=-2.2964


## Part 2: California Housing
8-qubit serial circuit, 1 FM per qubit, hardware-efficient triangle SU(4) ansatz
(6 layers × 7 nearest-neighbour SU(4) blocks, measurement on central qubit = wire 3)

In [6]:
# ── Data preparation ───────────────────────────────────────────
cal    = fetch_california_housing()
X_raw, y_raw = cal.data, cal.target

scaler_X = MinMaxScaler(feature_range=(-np.pi, np.pi))
scaler_y = MinMaxScaler(feature_range=(-1, 1))
X_scaled = scaler_X.fit_transform(X_raw)
y_scaled = scaler_y.fit_transform(y_raw.reshape(-1, 1)).ravel()

X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=SEED
)
X_tr = jnp.array(X_tr); X_te = jnp.array(X_te)
y_tr = jnp.array(y_tr); y_te = jnp.array(y_te)

print(f"California Housing: {len(X_tr)} train, {len(X_te)} test samples, "
      f"{X_raw.shape[1]} features")

California Housing: 16512 train, 4128 test samples, 8 features


In [7]:
# ── Circuit ────────────────────────────────────────────────────
# 8-qubit serial, 5 FMs per qubit → 6 serial ansatz layers
# Hardware-efficient triangle ansatz: 6 layers × 7 nearest-neighbour SU(4) gates
# Measurement on wire 3 (central qubit)

CH_WIRES         = 8
CH_FM_PER_QUBIT  = 5
CH_SERIAL_LAYERS = CH_FM_PER_QUBIT + 1  # 6 ansatz blocks
CH_ROT_PARAMS    = 15   # SU(4)
CH_SU_GATES      = 7   # 7 nearest-neighbour pairs on 8 qubits
CH_MSMT_WIRE     = 3

# Nearest-neighbour pairs for triangle ansatz
NN_PAIRS = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7)]

def S_8d(alpha, x, num_wires, enc_layer):
    """Each qubit gets its own feature and prefactor."""
    for w in range(num_wires):
        qml.RX(alpha[w] * x[w], wires=w)

def W_triangle(theta, t_blocks):
    """6 layers of 7 nearest-neighbour SU(4) blocks."""
    for layer in range(t_blocks):
        for pair_idx, (wa, wb) in enumerate(NN_PAIRS):
            qml.SpecialUnitary(theta[layer][pair_idx], wires=[wa, wb])

dev_ch = qml.device("default.qubit", wires=CH_WIRES)

@qml.qnode(dev_ch, interface="jax")
def circuit_ch(alpha, weights_SU, x=None):
    # Initial ansatz
    W_triangle(weights_SU[0], TRAINABLE_BLOCKS)
    for i in range(CH_SERIAL_LAYERS - 1):
        S_8d(alpha[i], x, CH_WIRES, i)
        W_triangle(weights_SU[i+1], TRAINABLE_BLOCKS)
    return qml.expval(qml.PauliZ(wires=CH_MSMT_WIRE))

circuit_ch_jit = jax.jit(circuit_ch)

# Alpha shape: (CH_FM_PER_QUBIT, CH_WIRES) — one row per FM layer
# Ternary: each qubit i gets prefactor 3^(i mod 5) per layer
TERNARY_ALPHA_CH = np.array([
    [3.0**j for j in range(CH_WIRES)]
    for _ in range(CH_FM_PER_QUBIT)
])
UNARY_ALPHA_CH = np.ones((CH_FM_PER_QUBIT, CH_WIRES))

# Weights shape: (CH_SERIAL_LAYERS, TRAINABLE_BLOCKS, CH_SU_GATES, CH_ROT_PARAMS)
def ch_weights_ones():
    return np.ones((CH_SERIAL_LAYERS, TRAINABLE_BLOCKS, CH_SU_GATES, CH_ROT_PARAMS))

CH_APPROACHES = [
    ("Fixed Unary",      UNARY_ALPHA_CH,   True),
    ("Trainable Unary",  UNARY_ALPHA_CH * np.linspace(1.0, 1.07, CH_WIRES), False),
    ("Fixed Ternary",    TERNARY_ALPHA_CH, True),
    ("Trainable Ternary",TERNARY_ALPHA_CH, False),
]

In [8]:
ch_results = []

for label, alpha_init, fixed in CH_APPROACHES:
    print(f"\n── California Housing  {label} ──")
    for run in range(NUM_RUNS):
        alpha   = jnp.array(alpha_init)
        weights = jnp.array(ch_weights_ones())

        if fixed:
            params = {"weights_SU": weights}
            def cost_fn(p, x_batch, y_batch):
                preds = [circuit_ch_jit(alpha, p["weights_SU"], x_)
                         for x_ in x_batch]
                return square_loss(y_batch, preds).squeeze()
        else:
            params = {"weights_SU": weights, "alpha": alpha}
            def cost_fn(p, x_batch, y_batch):
                preds = [circuit_ch_jit(p["alpha"], p["weights_SU"], x_)
                         for x_ in x_batch]
                return square_loss(y_batch, preds).squeeze()

        opt       = optax.adam(LR)
        opt_state = opt.init(params)

        @jax.jit
        def update(p, s, x, y):
            loss, g = jax.value_and_grad(cost_fn)(p, x, y)
            u, s2   = opt.update(g, s, p)
            return optax.apply_updates(p, u), s2, loss

        for step in range(MAX_STEPS):
            idx = np.random.randint(0, len(X_tr), BATCH_SIZE)
            params, opt_state, c = update(
                params, opt_state, X_tr[idx], y_tr[idx]
            )

        fa    = params.get("alpha", alpha)
        preds = [float(circuit_ch_jit(fa, params["weights_SU"], x_))
                 for x_ in X_te]
        r2    = r2_score(y_te, preds)
        print(f"  run={run+1}  R²={r2:.4f}")
        ch_results.append({"Dataset": "CaliforniaHousing", "Approach": label,
                            "Run": run, "R2_Score": r2})

ch_df = pd.DataFrame(ch_results)
ch_df.to_csv(OUT_DIR / "california_housing.csv", index=False)


── California Housing  Fixed Unary ──


E0515 08:26:06.631696 11485105 slow_operation_alarm.cc:73] 
********************************
[Compiling module jit_update for CPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
E0515 08:34:58.210714 11485103 slow_operation_alarm.cc:140] The operation took 10m51.584757s

********************************
[Compiling module jit_update for CPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************


  run=1  R²=-0.0738
  run=2  R²=0.0149

── California Housing  Trainable Unary ──
  run=1  R²=-0.0356


E0515 09:08:40.948177 11485105 slow_operation_alarm.cc:73] 
********************************
[Compiling module jit_update for CPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
E0515 09:21:23.407607 11485103 slow_operation_alarm.cc:140] The operation took 14m42.461647s

********************************
[Compiling module jit_update for CPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************


  run=2  R²=0.0086

── California Housing  Fixed Ternary ──
  run=1  R²=-0.0407


E0515 09:38:03.699768 11485105 slow_operation_alarm.cc:73] 
********************************
[Compiling module jit_update for CPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
E0515 09:46:20.483448 11485103 slow_operation_alarm.cc:140] The operation took 10m16.789812s

********************************
[Compiling module jit_update for CPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************


  run=2  R²=-0.0407

── California Housing  Trainable Ternary ──


E0515 09:50:39.075001 11485105 slow_operation_alarm.cc:73] 
********************************
[Compiling module jit_update for CPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
E0515 10:02:08.526660 11485103 slow_operation_alarm.cc:140] The operation took 13m29.459714s

********************************
[Compiling module jit_update for CPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************


  run=1  R²=-0.0373


KeyboardInterrupt: 

## Plot — fig:r2s_realworld

In [ ]:
from paper_style import boxplot_panel

fp_df = pd.read_csv(OUT_DIR / "flight_passengers.csv")
ch_df = pd.read_csv(OUT_DIR / "california_housing.csv")

approach_order  = ["Fixed Unary", "Trainable Unary",
                   "Fixed Ternary", "Trainable Ternary"]
approach_colors = [RED, BLUE, ORANGE, GREEN]
short_labels    = [r"Fixed\\Unary", r"Train.\\Unary",
                   r"Fixed\\Ternary", r"Train.\\Ternary"]

data_fp = {l: fp_df[fp_df.Approach==l].R2_Score.values for l in approach_order}
data_ch = {l: ch_df[ch_df.Approach==l].R2_Score.values for l in approach_order}

fig, axes = plt.subplots(1, 2, figsize=(6.8, 3.0))
plt.subplots_adjust(wspace=0.15)

boxplot_panel(axes[0], data_fp, approach_colors, show_ylabel=True)
axes[0].set_xticklabels(short_labels, fontsize=7)

boxplot_panel(axes[1], data_ch, approach_colors, show_ylabel=False)
axes[1].set_xticklabels(short_labels, fontsize=7)

# Dataset labels
axes[0].text(0.05, 0.97, "Flight Passengers",
             transform=axes[0].transAxes, fontsize=8,
             va="top", ha="left", style="italic")
axes[1].text(0.05, 0.97, "California Housing",
             transform=axes[1].transAxes, fontsize=8,
             va="top", ha="left", style="italic")

plt.savefig("r2s_realworld.pdf", dpi=600, bbox_inches="tight")
plt.savefig("r2s_realworld.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved r2s_realworld.pdf")

## Results table

In [ ]:
for dataset_name, df in [("Flight Passengers", fp_df),
                          ("California Housing", ch_df)]:
    print(f"\n{dataset_name}:")
    print(f"{'Approach':<22} {'N':>4} {'Median':>7} {'IQR':>18}")
    print("-" * 54)
    for a in approach_order:
        v   = df[df.Approach==a].R2_Score.values
        q25 = np.percentile(v, 25)
        q75 = np.percentile(v, 75)
        print(f"{a:<22} {len(v):>4} {np.median(v):>7.3f} "
              f"[{q25:5.3f},{q75:5.3f}]")